# 02 — Dari MiniSEED vertikal ke kandidat kurva dispersi SPAC

**Posisi dalam alur:** notebook 01 menyiapkan data 3 komponen, koordinat, dan HVSR; notebook ini memakai empat kanal vertikal dari folder rekaman yang sama dan koordinat yang disimpan otomatis oleh 01. Keluaran `dispersion_auto.csv` dan `dispersion_final.csv` menjadi masukan notebook 03. Jalankan 01 dulu setiap kali koordinat atau folder rekaman berubah.

**Yang Anda isi di sel berikut:** `SITE_ID`, parameter frekuensi/window/grid kecepatan, pilihan pick yang sudah ditinjau, serta status verifikasi geometri, jam, dan mode. Koordinat cukup diisi sekali di notebook 01. Anda tidak perlu mengedit CSV keluaran atau berkas konfigurasi site. Daftar pick dibiarkan kosong sampai kurva, enam pasangan sensor, dan ambiguitas cabang diperiksa.

**Teori singkat:** pada asumsi medan gelombang acak yang mendekati isotropik, koefisien SPAC untuk jarak antarsensor `r` dan frekuensi `f` mendekati `J₀(2πfr/c)`, dengan `c` kecepatan fase. Notebook menghitung bagian real koherensi spektral tiap pasangan, mengelompokkan jendela bersih, lalu mencari `c` yang cocok pada grid. Fungsi Bessel `J₀` berosilasi; beberapa kecepatan dapat cocok pada frekuensi yang sama. Karena itu hasil otomatis tetap kandidat sampai cabang dispersi, aperture array, dan mode gelombang direview.

**Keluaran:** koherensi per pasangan, grafik kecocokan J₀, QC window, kandidat otomatis, serta CSV pick terpilih. Tanpa bukti lapangan, `GEOMETRY_VERIFIED`, `PHYSICAL_CLOCK_DRIFT_VERIFIED`, dan `WAVEFIELD_MODE_VERIFIED` harus tetap `False`; notebook 03 hanya memakai hasil sebagai preview.

**Acuan MAM:** [Hayashi et al. (2022)](https://doi.org/10.1007/s10950-021-10051-y), terutama §6–7. Komponen HVSR/QC SESAME tetap memakai acuan khususnya.


## Input pengguna — parameter SPAC dan keputusan review

`MANUAL_ACCEPTED_PICKS` berisi baris seperti `{"frequency_hz": 12.0, "velocity_m_s": 350.0, "reason": "cabang kontinu dan pasangan konsisten"}`. Gunakan frekuensi persis dari `dispersion_auto.csv`; angka contoh ini **bukan** pick Solo. Jika sebuah status verifikasi diubah menjadi `True`, isi catatan buktinya. Perubahan pada sel ini otomatis menghasilkan ulang `dispersion_final.csv`.
`maximum_wavelength_ratio=None` hanya menambahkan diagnostik; batas numerik lama tetap berlaku. Isi angka positif hanya bila cutoff λ/r_max didukung data site. `WAVELENGTH_RANGE_REVIEWED` dan catatan buktinya wajib untuk inversi final. Figure 9 menunjukkan contoh bias pada 1,5–2× ukuran array, sedangkan §6.2 memberi pedoman umum 2–4×; keduanya bukan jaminan kelayakan.


In [ ]:
SITE_ID = "solo_pilot"
SPAC_PARAMETERS = {
    "resampled_rate_hz": 200,
    "window_seconds": 60,
    "frequency_min_hz": 1.0,
    "frequency_max_hz": 30.0,
    "frequency_count": 100,
    "log_band_half_width": 0.06,
    "velocity_min_m_s": 80.0,
    "velocity_max_m_s": 1500.0,
    "velocity_grid_count": 500,
    "maximum_wavelength_ratio": None,  # optional project cutoff, not a universal paper limit
}
MANUAL_ACCEPTED_PICKS = []
GEOMETRY_VERIFIED = False
PHYSICAL_CLOCK_DRIFT_VERIFIED = False
WAVEFIELD_MODE_VERIFIED = False
WAVELENGTH_RANGE_REVIEWED = False
VERIFICATION_NOTES = {"geometry": "", "physical_clock_drift": "", "wavefield_mode": "", "wavelength_range": ""}

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import UTC, datetime
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import scipy
from IPython.display import display
from scipy.signal import detrend, resample_poly, windows
from scipy.special import j0

from mhvsr_vs30.mam.pilot import center_on_outer_centroid, pair_distances
from mhvsr_vs30.mam.picking import (create_dispersion_picker, prepare_review_table,
    review_context_hash)
from mhvsr_vs30.mam.spac import fit_bessel_grid, real_coherency
from mhvsr_vs30.mam.wavelength import wavelength_diagnostics, depth_guidelines

ROOT = Path.cwd()
SITE_INPUTS = ROOT / 'outputs' / SITE_ID / '01' / 'site_inputs.json'
CONFIG = json.loads(SITE_INPUTS.read_text(encoding='utf-8'))
assert CONFIG['site_id'] == SITE_ID
RAW = ROOT / CONFIG['raw_directory']
PARAM = SPAC_PARAMETERS
OUT = ROOT / 'outputs' / SITE_ID / '02'
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
assert RAW.is_dir(), f'MiniSEED folder missing: {RAW}'
measured = {name: tuple(map(float, xy)) for name, xy in CONFIG['measured_coordinates'].items()}
modeled = center_on_outer_centroid(measured, 'Solo1', ('Solo2', 'Solo3', 'Solo4'))
geometry_choice = CONFIG['geometry_used_for_spac']
assert geometry_choice in {'triangle_hypothesis', 'supplied'}
active_coordinates = modeled if geometry_choice == 'triangle_hypothesis' else measured
PAIR_WITH_SOLO1_GROUP = 'center_outer' if geometry_choice == 'triangle_hypothesis' else 'solo1_other'
stations = list(measured)
pair_indices = list(combinations(range(len(stations)), 2))
pair_names = [(stations[a], stations[b]) for a, b in pair_indices]
distance_lookup = {(a, b): r for a, b, r in pair_distances(active_coordinates)}
distances = np.array([distance_lookup[pair] for pair in pair_names])
print('Stations:', stations, '| pairs:', pair_names)
(OUT/'dispersion_qc.json').write_text(json.dumps({'run_state':'running',
    'method_revision':'hayashi_2022_v1'}),encoding='utf-8')


## 1. Baca MiniSEED vertikal dan selaraskan rentang waktu

Header dipakai untuk menentukan irisan waktu bersama. Indeks sampel dipotong pada grid yang sama; data kemudian diturunkan dari 2.000 ke 200 Hz dengan filter anti-alias `resample_poly`. Ini memeriksa keselarasan timestamp, belum memeriksa drift jam fisik atau jejak penghapusan respons pada file `*_corr.mseed`.


In [ ]:
files = {}
headers = {}
for station in stations:
    candidates = sorted(RAW.glob(f'{station}_GHZ_*.mseed'))
    assert len(candidates) == 1, f'{station}: expected exactly one GHZ MiniSEED, got {len(candidates)}'
    path = candidates[0]
    header = obspy.read(str(path), headonly=True)
    assert len(header) == 1 and header[0].stats.channel == 'GHZ'
    files[station], headers[station] = path, header[0].stats
raw_rates = {float(h.sampling_rate) for h in headers.values()}
assert len(raw_rates) == 1, f'sampling rates differ: {raw_rates}'
raw_hz = raw_rates.pop()
target_hz = float(PARAM['resampled_rate_hz'])
factor = raw_hz / target_hz
assert factor.is_integer() and factor >= 1, 'resampling requires an integer downsampling factor'
factor = int(factor)
common_start = max(h.starttime for h in headers.values())
common_end = min(h.endtime for h in headers.values())
assert common_end > common_start
raw_samples = int(np.floor(float(common_end - common_start) * raw_hz))
raw_samples -= raw_samples % factor
assert raw_samples > int(PARAM['window_seconds'] * raw_hz)
data, input_rows = [], []
for station in stations:
    path = files[station]
    trace = obspy.read(str(path))[0]
    offset_float = float(common_start - trace.stats.starttime) * raw_hz
    offset = int(round(offset_float))
    assert abs(offset_float - offset) < 1e-4, f'{station}: timestamp off sample grid'
    section = np.asarray(trace.data[offset:offset + raw_samples], dtype=np.float64)
    assert len(section) == raw_samples and np.isfinite(section).all(), f'{station}: invalid samples'
    reduced = resample_poly(section, 1, factor)
    data.append(reduced)
    input_rows.append(dict(station=station, file=path.relative_to(ROOT).as_posix(),
                           sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                           station_id=trace.stats.station, raw_rate_hz=raw_hz,
                           sample_offset=offset, resampled_samples=len(reduced)))
aligned = np.stack(data)
assert aligned.shape[0] == 4 and aligned.shape[1] == raw_samples // factor
input_df = pd.DataFrame(input_rows)
input_df.to_csv(OUT / 'input_inventory.csv', index=False)
print('Common time:', common_start, 'to', common_start + raw_samples/raw_hz,
      '| duration s:', raw_samples/raw_hz, '| samples at 200 Hz:', aligned.shape[1])
print(input_df[['station','station_id','sample_offset','resampled_samples']].to_string(index=False))


## 2. QC window dan spektrum

Window 60 detik tanpa overlap. Window dengan amplitudo RMS sangat tinggi pada salah satu sensor ditandai dari median dan MAD log RMS. Ambang ini hanya menyaring transien besar; seluruh keputusan tersimpan. Spektrum memakai detrend linear dan taper Hann. Band frekuensi target dibentuk secara logaritmik dengan lebar relatif yang tercantum pada sel input pengguna.

In [ ]:
window_samples = int(PARAM['window_seconds'] * target_hz)
n_windows = aligned.shape[1] // window_samples
assert n_windows >= 8
chunks = aligned[:, :n_windows*window_samples].reshape(4,n_windows,window_samples).transpose(1,0,2)
rms = np.sqrt(np.mean(chunks**2,axis=2))
log_rms = np.log(np.maximum(rms, np.finfo(float).tiny))
med = np.median(log_rms,axis=0)
mad = np.median(np.abs(log_rms-med),axis=0)
scale = np.maximum(1.4826*mad, 0.1)
high_transient = log_rms > med + 4*scale
accepted = ~np.any(high_transient,axis=1)
assert accepted.sum() >= 8, 'too few clean windows for SPAC'
qc_df = pd.DataFrame({'window_index':np.arange(n_windows),
                      'start_utc':[str(common_start + i*PARAM['window_seconds']) for i in range(n_windows)],
                      'accepted':accepted,
                      'reason':['accepted' if ok else 'high_rms_transient' for ok in accepted]})
for k, station in enumerate(stations):
    qc_df[f'{station}_rms'] = rms[:,k]
    qc_df[f'{station}_high_transient'] = high_transient[:,k]
qc_df.to_csv(OUT / 'window_qc.csv',index=False)
taper = windows.hann(window_samples,sym=False)
spectra = np.fft.rfft(detrend(chunks[accepted],axis=-1,type='linear')*taper,axis=-1)
fft_frequency = np.fft.rfftfreq(window_samples,1/target_hz)
target_frequency = np.geomspace(PARAM['frequency_min_hz'],PARAM['frequency_max_hz'],
                                PARAM['frequency_count'])
half_width = float(PARAM['log_band_half_width'])
bands = [(int(np.searchsorted(fft_frequency,hz*np.exp(-half_width))),
          int(np.searchsorted(fft_frequency,hz*np.exp(half_width),side='right')))
         for hz in target_frequency]
assert all(0 <= a < b <= len(fft_frequency) for a,b in bands)
print('Windows:',n_windows,'accepted:',int(accepted.sum()),'frequency band:',
      target_frequency[0],'-',target_frequency[-1],'Hz')


## 3. Koherensi SPAC per pasangan dan variasi antarblok

Koherensi tiap pasangan dihitung dari spektrum silang rata-rata dibagi akar daya rata-rata kedua sensor. Bagian realnya menjadi koefisien kandidat SPAC. Simpangan baku antarblok 4 window dilaporkan sebagai indikator kestabilan, bukan interval keyakinan terkalibrasi. Pasangan pusat–tepi dan tepi–tepi ditampilkan terpisah, tanpa membaca ring Geopsy.

Normalisasi: rata-rata conj(Xi)Xj pada window diterima dan band frekuensi dibagi akar hasil kali rata-rata daya masing-masing sensor, lalu diambil bagian real. Fit menggunakan setiap jarak aktual dengan bobot pasangan sama; kategori pasangan pada grafik bukan penggabungan radius. Cakupan azimut tetap perlu direview. SEM blok adalah variasi coherency, bukan sigma kecepatan fase.


In [ ]:
coherency = real_coherency(spectra,pair_indices,bands)
block_size = 4
block_curves = [real_coherency(spectra[i:i+block_size],pair_indices,bands)
                for i in range(0,len(spectra),block_size) if len(spectra[i:i+block_size]) == block_size]
block_curves = np.stack(block_curves)
block_sem = np.std(block_curves,axis=0,ddof=1)/np.sqrt(len(block_curves))
assert np.isfinite(coherency).all() and np.nanmax(np.abs(coherency)) <= 1.000001
pair_rows = []
for j,(left,right) in enumerate(pair_names):
    group = PAIR_WITH_SOLO1_GROUP if 'Solo1' in (left,right) else 'outer_outer'
    for i,hz in enumerate(target_frequency):
        pair_rows.append(dict(frequency_hz=hz,station_a=left,station_b=right,
                              distance_m=distances[j],pair_group=group,
                              azimuth_deg=float(np.degrees(np.arctan2(
                                  active_coordinates[right][0]-active_coordinates[left][0],
                                  active_coordinates[right][1]-active_coordinates[left][1])) % 180),
                              real_coherency=coherency[i,j],block_sem=block_sem[i,j],
                              n_windows=int(accepted.sum()),n_full_blocks=len(block_curves)))
pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(OUT / 'spac_pairs.csv',index=False)
np.savez_compressed(OUT / 'spac_coefficients.npz',frequency_hz=target_frequency,
                    real_coherency=coherency,block_sem=block_sem,distances_m=distances,
                    station_a=np.array([a for a,b in pair_names]),
                    station_b=np.array([b for a,b in pair_names]),
                    accepted_windows=accepted)
print(pair_df[['station_a','station_b','distance_m','pair_group']].drop_duplicates()
      .groupby('pair_group').agg(pairs=('station_a','size'),
      min_distance_m=('distance_m','min'),max_distance_m=('distance_m','max')).to_string())

## 4. Pick otomatis J0 dengan QC numerik dan sensitivitas geometri

Satu kecepatan dipilih per frekuensi dari grid yang tercatat. Kurva J0 dapat mempunyai beberapa minimum dan pick antarfrekuensi belum dipaksa kontinu. Kolom `qc_reason` memisahkan kegagalan numerik; `automatic_status=provisional` tetap **belum** berarti pick final. Kecepatan final hanya diisi untuk baris yang Anda pilih pada `MANUAL_ACCEPTED_PICKS` di sel input. CSV yang dihasilkan adalah keluaran, bukan tempat mengisi keputusan.

In [ ]:
velocity_grid = np.geomspace(PARAM['velocity_min_m_s'],PARAM['velocity_max_m_s'],
                                  PARAM['velocity_grid_count'])
fit = fit_bessel_grid(target_frequency,distances,coherency,velocity_grid)
measured_lookup = {(a,b):r for a,b,r in pair_distances(measured)}
measured_distances = np.array([measured_lookup[pair] for pair in pair_names])
fit_supplied = fit_bessel_grid(target_frequency,measured_distances,coherency,velocity_grid)
candidate_rows=[]
for i,hz in enumerate(target_frequency):
    speed=float(fit['velocity_m_s'][i])
    rmse=float(fit['fit_rmse'][i])
    second=float(fit['second_minimum_rmse'][i])
    null=float(fit['null_rmse'][i])
    phase=float(2*np.pi*hz*max(distances)/speed)
    reasons=[]
    if not np.isfinite(speed) or speed <= 0:
        raise ValueError('No finite Bessel candidate; inspect SPAC before inversion')
    if fit['velocity_at_grid_bound'][i]: reasons.append('velocity_grid_boundary')
    if phase < 0.75: reasons.append('weak_aperture')
    if rmse > 0.18: reasons.append('poor_j0_fit')
    if np.isfinite(second) and second-rmse < 0.03: reasons.append('ambiguous_branch')
    if null-rmse < 0.04: reasons.append('weak_improvement_over_flat')
    if np.nanmedian(block_sem[i]) > 0.15: reasons.append('unstable_time_blocks')
    candidate_rows.append(dict(frequency_hz=hz,period_s=1/hz,
        velocity_auto_m_s=speed,fit_rmse=rmse,second_minimum_rmse=second,
        null_rmse=null,maximum_kr=phase,median_block_sem=float(np.nanmedian(block_sem[i])),
        velocity_supplied_geometry_m_s=float(fit_supplied['velocity_m_s'][i]),
        geometry_velocity_difference_m_s=float(speed-fit_supplied['velocity_m_s'][i]),
        qc_reason=';'.join(reasons) if reasons else 'numerical_checks_passed',
        automatic_status='rejected_numeric' if reasons else 'provisional'))
auto_df=pd.DataFrame(candidate_rows)
diagnostics=wavelength_diagnostics(target_frequency,auto_df.velocity_auto_m_s.to_numpy(),
    float(max(distances)),maximum_wavelength_ratio=PARAM['maximum_wavelength_ratio'])
for key in ('wavelength_m','guide_depth_m','wavelength_to_aperture','within_wavelength_limit'):
    auto_df[key]=diagnostics[key]
auto_df['aperture_warning']=np.where(auto_df.wavelength_to_aperture>4,
    'beyond_general_2_to_4_guideline',np.where(auto_df.wavelength_to_aperture>2,
    'review_against_site_bias','no_ratio_warning_not_proof_of_resolution'))
outside=~auto_df.within_wavelength_limit
auto_df.loc[outside,'automatic_status']='rejected_numeric'
auto_df.loc[outside,'qc_reason']=auto_df.loc[outside,'qc_reason']+';project_wavelength_limit'
auto_df.to_csv(OUT/'wavelength_diagnostics.csv',index=False)
auto_df.to_csv(OUT/'dispersion_auto.csv',index=False)
final_path=OUT/'dispersion_final.csv'
review_path=OUT/'dispersion_review.json'
review_context=review_context_hash(SITE_INPUTS,OUT/'input_inventory.csv',OUT/'dispersion_auto.csv')
final_df=prepare_review_table(auto_df,final_path,review_path,review_context,MANUAL_ACCEPTED_PICKS)
verification_flags={'geometry':GEOMETRY_VERIFIED,
    'physical_clock_drift':PHYSICAL_CLOCK_DRIFT_VERIFIED,
    'wavefield_mode':WAVEFIELD_MODE_VERIFIED,
    'wavelength_range':WAVELENGTH_RANGE_REVIEWED}
for name,verified in verification_flags.items():
    if verified and not VERIFICATION_NOTES[name].strip():
        raise ValueError(f'Add evidence to VERIFICATION_NOTES[{name!r}] before marking it verified')
accepted_manual=final_df.loc[final_df.accepted_for_inversion]
if len(accepted_manual):
    manual_diagnostic=wavelength_diagnostics(accepted_manual.frequency_hz.to_numpy(),
        accepted_manual.velocity_final_m_s.to_numpy(),float(max(distances)),
        maximum_wavelength_ratio=PARAM['maximum_wavelength_ratio'])
    if not manual_diagnostic['within_wavelength_limit'].all():
        raise ValueError('Manual pick exceeds project wavelength limit; review the selected band')
final_df.to_csv(final_path,index=False)
print(auto_df.automatic_status.value_counts().to_string())
print('Auto velocity range (m/s):',float(auto_df.velocity_auto_m_s.min()),
      'to',float(auto_df.velocity_auto_m_s.max()))

## 5. Gambar dan metadata keputusan

Panel SPAC menampilkan semua pasangan, bukan hanya rata-rata ring, agar ketidaksesuaian azimut tampak. Panel dispersi menunjukkan kandidat otomatis serta pengaruh pilihan geometri. Tidak ada kecepatan yang dinyatakan final pada tahap ini.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5),sharex=True,sharey=True)
for ax,group in zip(axes,(PAIR_WITH_SOLO1_GROUP,'outer_outer')):
    for j,(left,right) in enumerate(pair_names):
        if (PAIR_WITH_SOLO1_GROUP if 'Solo1' in (left,right) else 'outer_outer') != group: continue
        ax.semilogx(target_frequency,coherency[:,j],lw=1,label=f'{left}-{right} ({distances[j]:.2f} m)')
    ax.axhline(0,color='k',lw=0.6)
    ax.set(title=group,xlabel='Frequency (Hz)',ylabel='Real coherency',ylim=(-0.55,1.05))
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(FIG/'spac_pairs.png',dpi=160); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for j,(left,right) in enumerate(pair_names):
    axes[0].semilogx(target_frequency,coherency[:,j],alpha=0.55,lw=0.8)
for hz in (5,10,20):
    i=int(np.argmin(np.abs(target_frequency-hz)))
    axes[1].scatter(distances,coherency[i],label=f'{target_frequency[i]:.1f} Hz')
    x=np.linspace(0,max(distances)*1.1,200)
    axes[1].plot(x,j0(2*np.pi*target_frequency[i]*x/fit['velocity_m_s'][i]),lw=1)
axes[0].set(xlabel='Frequency (Hz)',ylabel='Real coherency',title='All six pairs')
axes[1].set(xlabel='Pair distance (m)',ylabel='Real coherency',title='Example J0 grid fits')
for ax in axes: ax.grid(alpha=0.2)
axes[1].legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG/'j0_fit_qc.png',dpi=160); plt.show()

fig,ax=plt.subplots(figsize=(8,4.5))
good=auto_df.automatic_status.eq('provisional').to_numpy()
ax.semilogx(target_frequency,fit['velocity_m_s'],color='0.7',lw=1,label='Auto all')
ax.scatter(target_frequency[good],fit['velocity_m_s'][good],s=12,label='Numerical QC passed')
ax.semilogx(target_frequency,fit_supplied['velocity_m_s'],lw=1,alpha=0.7,
            label='Using supplied geometry')
ax.set(xlabel='Frequency (Hz)',ylabel='Candidate phase velocity (m/s)',
       title='Automatic candidates — geometry and mode unverified')
ax.grid(alpha=0.2); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIG/'dispersion_auto_qc.png',dpi=160); plt.show()

# Phase-velocity image: RMS of J0 residuals over actual pair distances.
image_error=np.sqrt(np.mean((j0(2*np.pi*target_frequency[:,None,None]*
    distances[None,:,None]/velocity_grid[None,None,:])-coherency[:,:,None])**2,axis=1))
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
mesh=axes[0].pcolormesh(target_frequency,velocity_grid,image_error.T,shading='auto',cmap='viridis_r')
axes[0].plot(target_frequency,fit['velocity_m_s'],'r.',ms=3,label='Automatic candidates')
axes[0].set(xscale='log',yscale='log',xlabel='Frequency (Hz)',ylabel='Phase velocity (m/s)',title='J0 residual image')
axes[0].legend(fontsize=8); fig.colorbar(mesh,ax=axes[0],label='RMS coherency residual')
axes[1].semilogx(target_frequency,auto_df.wavelength_to_aperture,label='Candidate wavelength / r_max')
axes[1].axhspan(2,4,color='0.8',alpha=0.5,label='General guideline, not acceptance band')
if PARAM['maximum_wavelength_ratio'] is not None:
    axes[1].axhline(PARAM['maximum_wavelength_ratio'],color='r',ls='--',label='Project cutoff')
axes[1].set(xlabel='Frequency (Hz)',ylabel='Wavelength / max receiver spacing',title='Aperture diagnostic')
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.2)
fig.tight_layout(); fig.savefig(FIG/'phase_velocity_wavelength_qc.png',dpi=160); plt.show()

accepted_final=final_df.loc[final_df.accepted_for_inversion]
depth_final=(depth_guidelines(accepted_final.frequency_hz.to_numpy(),
    accepted_final.velocity_final_m_s.to_numpy()) if len(accepted_final) and
    all(verification_flags.values()) else None)
metadata={
    'run_state':'complete','method_revision':'hayashi_2022_v1',
    'primary_reference_doi':'10.1007/s10950-021-10051-y',
    'maximum_receiver_spacing_m':float(max(distances)),
    'wavelength_range_reviewed':WAVELENGTH_RANGE_REVIEWED,
    'depth_guidelines_reviewed_input_m':depth_final,
    'depth_guidelines_are_resolution_proof':False,
    'coherency_averaging':'pooled_cross_and_auto_spectra_over_windows_and_band',
    'spac_fit':'equal_pair_weight_actual_distances_no_radius_pooling',
    'dispersion_auto_sha256':hashlib.sha256((OUT/'dispersion_auto.csv').read_bytes()).hexdigest(),
    'dispersion_final_sha256':hashlib.sha256((OUT/'dispersion_final.csv').read_bytes()).hexdigest(),
    'review_context_sha256':review_context,
    'site_id':CONFIG['site_id'],'created_at_utc':datetime.now(UTC).isoformat(),
    'input_role':'four_vertical_miniseed_plus_user_coordinates',
    'external_processed_outputs_used':False,
    'input_files':input_df[['file','sha256','station_id']].to_dict('records'),
    'python_version':platform.python_version(),'obspy_version':obspy.__version__,
    'numpy_version':np.__version__,'scipy_version':scipy.__version__,
    'processing_parameters':PARAM,'raw_rate_hz':raw_hz,'resampled_rate_hz':target_hz,
    'common_start_utc':str(common_start),'common_duration_s':raw_samples/raw_hz,
    'n_windows':n_windows,'n_accepted_windows':int(accepted.sum()),
    'n_full_blocks':len(block_curves),'coordinate_reference_system':CONFIG['coordinate_reference_system'],
    'geometry_used':geometry_choice,'geometry_verified':GEOMETRY_VERIFIED,
    'physical_clock_drift_verified':PHYSICAL_CLOCK_DRIFT_VERIFIED,
    'wavefield_mode_verified':WAVEFIELD_MODE_VERIFIED,
    'verification_notes':VERIFICATION_NOTES,
    'site_inputs_sha256':hashlib.sha256(SITE_INPUTS.read_bytes()).hexdigest(),
    'response_correction_lineage_verified':False,
    'n_provisional_numeric_picks':int(good.sum()),
    'n_final_accepted_picks':int(final_df.accepted_for_inversion.sum()),
    'blocking_review':['true sensor positions','instrument clock drift','wavefield isotropy and mode',
                       'manual dispersion pick review','wavelength range review'],
}
qc_path=OUT/'dispersion_qc.json'
qc_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
picker=create_dispersion_picker(auto_df,final_df,pair_names,distances,coherency,block_sem,
    block_curves,velocity_grid,final_path,review_path,review_context,qc_path,
    context_paths=(SITE_INPUTS,OUT/'input_inventory.csv',OUT/'dispersion_auto.csv'))
display(picker)
print('Saved:',OUT,'| final accepted:',metadata['n_final_accepted_picks'])

## Review sebelum inversi

Tinjau posisi sensor sebenarnya, sistem koordinat, sinkronisasi jam fisik, bentuk keenam kurva pasangan, kesinambungan cabang J₀, serta kemungkinan mode gelombang. Kembali ke **sel input pengguna** di atas untuk mengisi `MANUAL_ACCEPTED_PICKS` dan catatan verifikasi, lalu jalankan notebook 02 dari awal. Tanpa bukti, biarkan daftar pick kosong dan status verifikasi `False`; notebook 03 tetap dapat membuat preview, tetapi tidak menghasilkan profil final.
Pedoman Hayashi Eq. (2): D_min≈λ_min/3 dan D_max≈λ_max/2 dihitung hanya untuk pick yang direview. Keduanya tidak otomatis membuktikan resolusi 0–30 m. MMSPAC/direct fitting (Figure 6) merupakan alternatif, belum diimplementasikan di sini. Optimasi grid ini belum mengidentifikasi higher modes.
